# Extended Data Figure 3 — Temporal trends in gene-measurement associations

Cumulative unique measurement-associated genes (top) and gene–measurement pairs (bottom)
discovered per year, as nested tiers of increasing inclusiveness:

1. **EUR common** — MAF $\geq 0.01$, studies where one ancestry reaches $\geq 90\%$ and it is
   non-Finnish European
2. **+ non-EUR common** — MAF $\geq 0.01$, studies where one ancestry reaches $\geq 90\%$ and it is
   **not** non-Finnish European (Finnish included)
3. **+ mixed common** — MAF $\geq 0.01$, studies where **no** single ancestry reaches $90\%$
   (pan-ancestry meta-analyses)
4. **+ rare** — MAF $< 0.01$, any ancestry

Each bar segment is the increment that tier adds over the tiers below it.

> **Revised for reviewer round 1.** The published version used a binary ancestry split
> (`nfe_common` / `non_nfe_common`), which put pan-ancestry meta-analyses and FinnGen in the same
> bucket as genuinely non-European studies. `mixed` is now its own layer. See
> `chapters/06-review-r1/ancestry-mixed-split/` for the reclassification and the numbers.

**Data:** `data/intermediate_files/l2g_measurements_full-r1.csv`, produced by
`chapters/06-review-r1/ancestry-mixed-split/01_ancestry_reclassification.ipynb`.
That notebook must be run first. No Spark session is needed here.

## Changelog — reviewer round 1

What changed relative to the published version of this notebook.

| # | Published version | Now |
| - | ----------------- | --- |
| 1 | **Three** stacked layers: `nfe_common`, `non_nfe_common`, rare | **Four** layers: `EUR common`, `non-EUR common`, `mixed common`, rare |
| 2 | Ancestry was binary — anything not $\geq 90\%$ non-Finnish European counted as "non-EUR", so pan-ancestry meta-analyses and FinnGen sat with genuinely non-European studies | `mixed` (no single ancestry $\geq 90\%$) is its own layer; `non-EUR` now means one ancestry $\geq 90\%$ and it is not NFE, Finnish included |
| 3 | Read `list_of_prioritised_genes_per_CS_with_year_nfe_maf.parquet` through a 40 GB PySpark session | Reads `l2g_measurements_full-r1.csv` with pandas — no Spark, no Java, runs in seconds |
| 4 | Layer heights computed inline with no verification | Tier totals asserted equal to `ed3_cumulative_discovery_nested-r1.csv` from the R1 analysis notebook |
| 5 | `ffill().fillna(0)` on reindexed cumulative counts | Fixed year axis 2006–2024 with `cumsum` over per-year counts, so a year with no discoveries carries the previous total instead of dropping out |
| 6 | No numbers printed | Final-year tier totals, layer heights and layer percentages printed at the end |

**Unchanged:** panel layout (genes on top, gene–measurement pairs below), pair definition
(`geneId` $\times$ `studyId`), colours for the EUR / non-EUR / rare layers, MAF threshold 0.01,
ancestry threshold 90%, exclusion of 2025.

**Output file.** Written as `extended_figure_3-r1.pdf`, leaving the published
`extended_figure_3.pdf` in place, matching the `-r1` convention used for `Figure_1_combined-r1.pdf`.

**Effect on the figure.** Totals are identical to the published version — 15,061 genes and 358,449
pairs at 2024, and the EUR-common bottom layer is unchanged at 13,914 genes. Only the old single
"non-EUR" band splits in two: for genes, 723 (4.8%) genuinely non-EUR and 293 (2.0%) mixed; for
pairs, 30,553 (8.5%) non-EUR and 68,188 (19.0%) mixed.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

In [ ]:
path_to_intermediate_data_folder = str(paper.DERIVED) + "/"

measurements_path = path_to_intermediate_data_folder + "prioritised_genes_measurements"
reference_path = str(paper.BASELINE / "ed3_cumulative_discovery_nested-r1.csv")

MAX_YEAR = 2024  # 2025 is a partial year
FIRST_YEAR = 2006

# Nesting order of the stacked layers, bottom to top.
ANCESTRY_ORDER = ["EUR", "non-EUR", "mixed"]
ANCESTRY_COLORS = {"EUR": "#4472C4", "non-EUR": "#70AD47", "mixed": "#ED7D31", "rare": "#FFC000"}
LAYER_LABELS = {
    "EUR": r"EUR common (MAF $\geq$ 0.01)",
    "non-EUR": r"Non-EUR common (MAF $\geq$ 0.01)",
    "mixed": r"Mixed ancestry common (MAF $\geq$ 0.01)",
    "rare": r"Rare (MAF $<$ 0.01)",
}

## Load gene–measurement associations

In [ ]:
l2g_measurements = pd.read_parquet(
    measurements_path,
    columns=["studyLocusId", "studyId", "geneId", "year", "ancestryClass", "freqClass"],
)
print(f"gene-measurement rows: {len(l2g_measurements):,}")
print(f"unique genes:          {l2g_measurements['geneId'].nunique():,}")
l2g_measurements.groupby(["ancestryClass", "freqClass"]).size().rename("rows").reset_index()

## Cumulative discovery per nested tier

In [ ]:
def first_discovery(df: pd.DataFrame, id_cols: list[str], mask: pd.Series) -> pd.DataFrame:
    """First year each unique entity (gene, or gene-measurement pair) appears.

    Args:
        df: row-level gene-measurement table.
        id_cols: columns identifying the entity.
        mask: boolean mask restricting the rows considered.

    Returns:
        pandas.DataFrame: one row per entity with its first year of discovery.
    """
    sub = df[mask][id_cols + ["year"]].dropna(subset=["year"])
    sub = sub[sub["year"] <= MAX_YEAR].drop_duplicates()
    return sub.groupby(id_cols, as_index=False)["year"].min()


def cumulative_layers(df: pd.DataFrame, id_cols: list[str]) -> pd.DataFrame:
    """Cumulative count per nested tier, and the stacked layer heights between tiers.

    Args:
        df: row-level gene-measurement table.
        id_cols: columns identifying the entity being counted.

    Returns:
        pandas.DataFrame: indexed by year, one cumulative column per tier plus one layer
        column per stacked segment.
    """
    years = list(range(FIRST_YEAR, MAX_YEAR + 1))
    common = df["freqClass"] == "common"

    cumulative = pd.DataFrame(index=years)
    cumulative.index.name = "year"
    for k, ancestry in enumerate(ANCESTRY_ORDER, start=1):
        mask = common & df["ancestryClass"].isin(ANCESTRY_ORDER[:k])
        first = first_discovery(df, id_cols, mask)
        cumulative[ancestry] = first.groupby("year").size().reindex(years, fill_value=0).cumsum()
    first_all = first_discovery(df, id_cols, df["freqClass"].notna())
    cumulative["rare"] = first_all.groupby("year").size().reindex(years, fill_value=0).cumsum()

    layers = cumulative.diff(axis=1)
    layers[ANCESTRY_ORDER[0]] = cumulative[ANCESTRY_ORDER[0]]
    return cumulative.join(layers.add_prefix("layer_"))


genes = cumulative_layers(l2g_measurements, ["geneId"])
pairs = cumulative_layers(l2g_measurements, ["geneId", "studyId"])
genes

In [ ]:
# Cross-check against the tier totals computed in the R1 analysis notebook.
reference = pd.read_csv(reference_path)
for metric, table in [("measurement genes", genes), ("gene-measurement pairs (studyId)", pairs)]:
    expected = (
        reference[reference["metric"] == metric]
        .pivot_table(index="year", columns="tier_index", values="cumulative")
        .sort_index(axis=1)
    )
    expected.columns = ANCESTRY_ORDER + ["rare"]
    pd.testing.assert_frame_equal(
        table[ANCESTRY_ORDER + ["rare"]].astype(int),
        expected.loc[table.index].astype(int),
        check_names=False,
    )
    print(f"{metric}: matches the R1 analysis notebook")

## Extended Data Figure 3

In [ ]:
def make_stacked_bar_panel(ax: plt.Axes, table: pd.DataFrame, ylabel: str) -> None:
    """Stacked bar chart of cumulative discovery: EUR / non-EUR / mixed / rare layers.

    Args:
        ax: axes to draw on.
        table: output of `cumulative_layers`.
        ylabel: y-axis label.
    """
    x = [str(year) for year in table.index]
    bottom = np.zeros(len(table))
    for layer in ANCESTRY_ORDER + ["rare"]:
        height = table[f"layer_{layer}"].to_numpy()
        ax.bar(x, height, bottom=bottom, color=ANCESTRY_COLORS[layer], label=LAYER_LABELS[layer])
        bottom = bottom + height

    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax.grid(axis="y", color="lightgray", linestyle="--", linewidth=0.5, alpha=0.7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))

make_stacked_bar_panel(ax1, genes, "Cumulative unique genes")
ax1.set_title("Measurements associated genes", fontsize=10, fontweight="bold", loc="left")
ax1.set_xlabel("")

make_stacked_bar_panel(ax2, pairs, "Cumulative number of pairs")
ax2.set_title("Unique gene\u2013measurement pairs", fontsize=10, fontweight="bold", loc="left")
ax2.set_xlabel("Year", fontsize=9)

# Legend above both panels
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0.9), ncols=1)

fig.tight_layout()
fig.savefig(f"{figure_dir}/extended_figure_3.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Numbers behind the figure

In [ ]:
summary = pd.DataFrame(
    {
        "genes_cumulative": genes.loc[MAX_YEAR, ANCESTRY_ORDER + ["rare"]],
        "genes_layer": genes.loc[MAX_YEAR, [f"layer_{a}" for a in ANCESTRY_ORDER + ["rare"]]].to_numpy(),
        "pairs_cumulative": pairs.loc[MAX_YEAR, ANCESTRY_ORDER + ["rare"]],
        "pairs_layer": pairs.loc[MAX_YEAR, [f"layer_{a}" for a in ANCESTRY_ORDER + ["rare"]]].to_numpy(),
    }
).astype(int)
summary["genes_layer_pct"] = (100 * summary["genes_layer"] / genes.loc[MAX_YEAR, "rare"]).round(2)
summary["pairs_layer_pct"] = (100 * summary["pairs_layer"] / pairs.loc[MAX_YEAR, "rare"]).round(2)
summary